<a href="https://colab.research.google.com/github/oselumeseagbonrofo/small-llm-experiments/blob/main/full_finetuning_gpt2_small.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Full Finetuning with GPT2-small to write manim code

## Load Dataset

In [1]:
from datasets import load_dataset
dataset = load_dataset("Edoh/manim_python")

README.md:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/135k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/11.3k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/599 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/51 [00:00<?, ? examples/s]

## Load Model tokenizer

In [2]:
from transformers import GPT2Tokenizer
model_name = "openai-community/gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

## Implement preprocessing function

In [3]:
def preprocess_data(examples):
 inputs = [
 f"Instruction: {instr}\nOutput: {out}"
 for instr, out in zip(examples["instruction"], examples["output"])
 ]

 tokenized = tokenizer(inputs, truncation=True, max_length=512,
                       padding="max_length")

 tokenized["labels"] = tokenized["input_ids"].copy()
 return tokenized

tokenized_datasets = dataset.map(preprocess_data,
 batched=True,
 remove_columns=dataset["train"].column_names)

Map:   0%|          | 0/599 [00:00<?, ? examples/s]

Map:   0%|          | 0/51 [00:00<?, ? examples/s]

## Initialise model

In [4]:
from transformers import GPT2LMHeadModel
def model_init():
 return GPT2LMHeadModel.from_pretrained(model_name, device_map='auto')

## Configure training arguments for hyperparameter search

In [5]:
from transformers import TrainingArguments
training_args = TrainingArguments(
 output_dir="./gpt2-manim-python-finetuned",
 eval_strategy="epoch",
 save_strategy="epoch",
 logging_strategy="steps",
 logging_steps=100,
 save_total_limit=2,
 load_best_model_at_end=True,
 metric_for_best_model="eval_loss",
 greater_is_better=False,
 fp16=True,
 report_to="none",
)

In [6]:
train_val_split = tokenized_datasets["train"].train_test_split(test_size=0.1)
tokenized_datasets["train"] = train_val_split["train"]
tokenized_datasets["validation"] = train_val_split["test"]

## Initialise trainer

In [7]:
from transformers import (
 Trainer,
 DataCollatorForLanguageModeling,
 EarlyStoppingCallback
)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer,
 mlm=False)
trainer = Trainer(
 model_init=model_init,
 args=training_args,
 train_dataset=tokenized_datasets["train"],
 eval_dataset=tokenized_datasets["validation"],
 data_collator=data_collator,
 callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

## Define hyperparameter tuning search space

In [8]:
def hp_space(trial):
 return {
 "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True),
 "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size",
  [2, 4, 8]),
 "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.3),
 "num_train_epochs": trial.suggest_int("num_train_epochs", 3, 6),
 "warmup_steps": trial.suggest_int("warmup_steps", 0, 500),
 "gradient_accumulation_steps": trial.suggest_categorical("gradient_accumulation_steps",
  [1, 2, 4]),
 }

## Run hyperparameter search

In [9]:
!pip install -q optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 20.9 MB/s eta 0:00:00


In [10]:
best_run = trainer.hyperparameter_search(
 direction="minimize",
 backend="optuna",
 n_trials=10,
 hp_space=hp_space,
 compute_objective=lambda metrics: metrics["eval_loss"],
)

[I 2026-08-17 13:26:32,307] A new study created in memory with name: no-name-0ccccb72-fe23-43a1-9190-f388c8ac258a


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,1.380768,0.235794
2,0.254273,0.171742
3,0.158803,0.156502
4,0.138548,0.153637
5,0.123854,0.151195


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[I 2026-08-17 13:53:40,889] Trial 8 pruned. 


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,No log,1.607640


[I 2026-08-17 13:54:15,109] Trial 9 pruned. 


## Configure trainer with best hyperparameters

In [11]:
for key, value in best_run.hyperparameters.items():
 setattr(training_args, key, value)
trainer = Trainer(
 model_init=model_init,
 args=training_args,
 train_dataset=tokenized_datasets["train"],
 eval_dataset=tokenized_datasets.get("validation"),
 data_collator=data_collator,
 callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [12]:
trainer.train()
trainer.save_model("./gpt2-manim-python-finetuned")
tokenizer.save_pretrained("./gpt2-manim-python-finetuned")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,2.191768,0.378593
2,0.405903,0.206775
3,0.211292,0.179138
4,0.175601,0.153153
5,0.147880,0.148465


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./gpt2-manim-python-finetuned/tokenizer_config.json',
 './gpt2-manim-python-finetuned/tokenizer.json')

## Testing fine-tuned model

In [13]:
import torch
model_dir = "./gpt2-manim-python-finetuned"
tokenizer = GPT2Tokenizer.from_pretrained(model_dir)
model = GPT2LMHeadModel.from_pretrained(model_dir)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [14]:
def generate_output(instruction, max_length=150, num_beams=5,
                    temperature=0.7, top_p=0.9, repetition_penalty=1.2):
  prompt = f"Instruction: {instruction}\nOutput:"
  input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

  generated_ids = model.generate(
 input_ids,
 max_length=max_length,
 num_beams=num_beams,
 temperature=temperature,
 top_p=top_p,
 repetition_penalty=repetition_penalty,
 do_sample=True,
 pad_token_id=tokenizer.eos_token_id,
 eos_token_id=tokenizer.eos_token_id,
 early_stopping=True,
 no_repeat_ngram_size=2,
  )

  generated_text = tokenizer.decode(generated_ids[0],
 skip_special_tokens=True)
  output_start = generated_text.find("Output:")
  if output_start != -1:
    output_text = generated_text[output_start + len("Output:"):].strip()
  else:
    output_text = generated_text.strip()
  return output_text

In [15]:
import csv
output_csv = "gpt2_manim_python_test_outputs.csv"
with open(output_csv, mode="w", newline="", encoding="utf-8") as csvfile:
 writer = csv.DictWriter(csvfile,
 fieldnames=["instruction", "reference_output", "generated_output"])
 writer.writeheader()

 for example in dataset['test']:
  instruction = example["instruction"]
  reference_output = example["output"]
  generated_output = generate_output(instruction)
  writer.writerow({
  "instruction": instruction,
  "reference_output": reference_output,
  "generated_output": generated_output,
  })

## Domain specific evaluation

### Check if generated code is valid Python syntax

In [16]:
import ast

def is_syntax_valid(code_str):
 try:
  ast.parse(code_str)
  return True, ""
 except SyntaxError as e:
  return False, str(e)

### Perform static analysis for Manim API

In [18]:
from ast import NodeVisitor
class ManimCodeAnalyzer(ast.NodeVisitor):
 def __init__(self):
  self.imports_manim = False
  self.scene_subclass_names = []
  self.play_calls = 0
  self.create_calls = 0
  self.errors = []

 # Verify if Manim package is imported
 def visit_Import(self, node):
  for alias in node.names:
    if alias.name == "manim":
      self.imports_manim = True
  self.generic_visit(node)

  # Check if Manim elements are imported via from...import...
  def visit_ClassDef(self, node):
    for base in node.bases:
      if isinstance(base, ast.Name) and base.id == "Scene":
        self.scene_subclass_names.append(node.name)
      elif isinstance(base, ast.Attribute):
        if base.attr == "Scene":
          self.scene_subclass_names.append(node.name)
    self.generic_visit(node)

  # Counts calls to play methos in Scene subclass and Manim Create initializations
  def visit_Call(self, node):
    if isinstance(node.func, ast.Attribute):
      if (isinstance(node.func.value, ast.Name)
      and node.func.value.id == "self"
      and node.func.attr == "play"):
        self.play_calls += 1
    if isinstance(node.func, ast.Name) and node.func.id == "Create":
      self.create_calls += 1

    self.generic_visit(node)

  def analyze_manim_code(code_str):
      analyzer = ManimCodeAnalyzer()
      try:
        tree = ast.parse(code_str)
      except SyntaxError as e:
        return {
                "syntax_valid": False,
                "syntax_error": str(e),
                "imports_manim": False,
                "scene_subclass_names": [],
                "play_calls": 0,
                "create_calls": 0,
                }
      analyzer.visit(tree)
      return {
 "syntax_valid": True,
 "syntax_error": None,
 "imports_manim": analyzer.imports_manim,
 "scene_subclass_names": analyzer.scene_subclass_names,
 "play_calls": analyzer.play_calls,
 "create_calls": analyzer.create_calls,
 }

### Example of analyzer in action

In [21]:
code_sample = """
from manim import *
class MyScene(Scene):
 def construct(self):
  circle = Circle()
  self.play(Create(circle))
"""

manim_code_Analyzer = ManimCodeAnalyzer()
#manim_code_Analyzer.analyze_manim_code(code_sample)

### Executing generated code workflow

In [37]:
!sudo apt-get install -y libpango1.0-dev libcairo2-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libcairo2-dev is already the newest version (1.16.0-5ubuntu2.1).
The following additional packages will be installed:
  libdatrie-dev libfribidi-dev libgraphite2-dev libharfbuzz-dev
  libharfbuzz-gobject0 libharfbuzz-icu0 libthai-dev pango1.0-tools
Suggested packages:
  libdatrie-doc libgraphite2-utils libpango1.0-doc libthai-doc graphicsmagick
The following NEW packages will be installed:
  libdatrie-dev libfribidi-dev libgraphite2-dev libharfbuzz-dev
  libharfbuzz-gobject0 libharfbuzz-icu0 libpango1.0-dev libthai-dev
  pango1.0-tools
0 upgraded, 9 newly installed, 0 to remove and 67 not upgraded.
Need to get 912 kB of archives.
After this operation, 5,790 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libdatrie-dev amd64 0.2.13-2 [19.7 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libfribidi-dev amd64 1.0.8-2ubuntu3.

In [ ]:
!pip install manim

In [49]:
def evaluate_manim_code(code_str, scene_class_name="CustomScene"):
  import os
  import tempfile
  with tempfile.TemporaryDirectory() as tmpdir:
    code_path = os.path.join(tmpdir, "generated_scene.py")
    with open(code_path, "w") as f:
      f.write(code_str)
    cmd = ["manim", "-ql", code_path, scene_class_name]
    import subprocess
    try:
      result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
      success = result.returncode == 0
      output = result.stdout + "\n" + result.stderr
    except subprocess.TimeoutExpired:
      success = False
      output = "Timeout expired during rendering."
    return success, output

In [50]:
success, output = evaluate_manim_code(code_sample,
                                      scene_class_name="MyScene")
print("Render success:", success)
print("Output:", output)

Render success: True
Output: Manim Community v0.21.0

[08/17/26 14:20:45] INFO     Animation 0 : Partial      scene_file_writer.py:192
                             movie file written in                              
                             '/content/media/videos/gen                         
                             erated_scene/480p15/partia                         
                             l_movie_files/MyScene/2118                         
                             183852_1644652507_22313245                         
                             7.mp4'                                             
                    INFO     Combining to Movie file.   scene_file_writer.py:952
                    INFO                               scene_file_writer.py:1103
                             File ready at                                      
                             '/content/media/videos/ge                          
                             nerated_scene/480p15/MySc 

### Note: Tips for validating execution and rendering correctness in tasks dealing with images



*   Sandboxing (using Docker containers or restricted environments) to isolate execution.

*    Set timeouts to prevent hangs or infinite loops.

*   Limit CPU, GPU, and memory usage during rendering to save time and money.

